# Bab 7. Objek dan Kelas Secukupnya

Kode pendamping buku *Python untuk Machine Learning dan Data
Science*. Jalankan selnya berurutan dari atas, sebab sebagian
sel memakai peubah dari sel sebelumnya.

Notebook ini dibangkitkan dari naskah buku. Jangan disunting di
sini, sunting listing pada berkas `.tex` lalu bangkitkan ulang.

## 1. Data dan fungsi terpisah

In [ ]:
nama = "Nastar"
harga = 85000
stok = 12

def nilai_stok(harga, stok):
    return harga * stok

print(nilai_stok(harga, stok))

## 2. Kelas pertama

In [ ]:
class Produk:
    def __init__(self, nama, harga, stok=0):
        self.nama = nama
        self.harga = harga
        self.stok = stok

    def nilai_stok(self):
        return self.harga * self.stok

p = Produk("Nastar", 85000, 12)
print(p.nama)
print(p.nilai_stok())

Keluaran yang diharapkan:

```
Nastar
1020000
```

## 3. Representasi yang berguna

In [ ]:
class Produk:
    def __init__(self, nama, harga, stok=0):
        self.nama = nama
        self.harga = harga
        self.stok = stok

    def __repr__(self):
        return (f"Produk(nama={self.nama!r}, "
                f"harga={self.harga})")

print(Produk("Nastar", 85000, 12))

Keluaran yang diharapkan:

```
Produk(nama='Nastar', harga=85000)
```

## 4. Kesalahan yang tampak wajar

In [ ]:
class Keranjang:
    isi = []                  # milik kelas

    def tambah(self, x):
        self.isi.append(x)

a = Keranjang()
b = Keranjang()
a.tambah("Nastar")

print(a.isi)
print(b.isi)

Keluaran yang diharapkan:

```
['Nastar']
['Nastar']
```

## 5. Perbaikannya

In [ ]:
class Keranjang:
    def __init__(self):
        self.isi = []         # milik instance

    def tambah(self, x):
        self.isi.append(x)

c = Keranjang()
d = Keranjang()
c.tambah("Nastar")

print(c.isi, d.isi)

Keluaran yang diharapkan:

```
['Nastar'] []
```

## 6. Argumen bawaan yang berbahaya

In [ ]:
def catat(item, daftar=[]):
    daftar.append(item)
    return daftar

print(catat("a"), catat("b"), catat("c"))

Keluaran yang diharapkan:

```
['a', 'b', 'c'] ['a', 'b', 'c'] ['a', 'b', 'c']
```

## 7. Pola None

In [ ]:
def catat(item, daftar=None):
    if daftar is None:
        daftar = []
    daftar.append(item)
    return daftar

print(catat("a"), catat("b"), catat("c"))

Keluaran yang diharapkan:

```
['a'] ['b'] ['c']
```

## 8. Membandingkan menurut isi

In [ ]:
class Produk:
    def __init__(self, nama, harga):
        self.nama = nama
        self.harga = harga

    def __eq__(self, lain):
        if not isinstance(lain, Produk):
            return NotImplemented
        return ((self.nama, self.harga)
                == (lain.nama, lain.harga))

u = Produk("Nastar", 85000)
v = Produk("Nastar", 85000)
print(u == v)

Keluaran yang diharapkan:

```
True
```

## 9. Objek yang berperilaku seperti list

In [ ]:
class Katalog:
    def __init__(self, produk):
        self.produk = list(produk)

    def __len__(self):
        return len(self.produk)

    def __getitem__(self, i):
        return self.produk[i]

    def __repr__(self):
        return f"Katalog({len(self)} produk)"

k = Katalog([Produk("Nastar", 85000),
             Produk("Brownies", 45000)])

print(k)
print(len(k))
print([q.nama for q in k])

Keluaran yang diharapkan:

```
Katalog(2 produk)
2
['Nastar', 'Brownies']
```

## 10. Menolak nilai yang mustahil

In [ ]:
class Produk:
    def __init__(self, nama, harga):
        self.nama = nama
        self.harga = harga

    @property
    def harga(self):
        return self._harga

    @harga.setter
    def harga(self, nilai):
        if nilai < 0:
            raise ValueError("harga tidak boleh negatif")
        self._harga = nilai

p = Produk("Nastar", 85000)
print(p.harga)
p.harga = -100

Keluaran yang diharapkan:

```
85000
ValueError: harga tidak boleh negatif
```

## 11. Dataclass

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Transaksi:
    produk: str
    jumlah: int
    harga: int
    catatan: list = field(default_factory=list)

    @property
    def omzet(self):
        return self.jumlah * self.harga

t1 = Transaksi("Nastar", 3, 85000)
t2 = Transaksi("Nastar", 3, 85000)

print(t1)
print(t1.omzet)
print(t1 == t2)

Keluaran yang diharapkan:

```
Transaksi(produk='Nastar', jumlah=3, harga=85000, catatan=[])
255000
True
```

## 12. Komposisi

In [ ]:
class Diskon:
    def __init__(self, persen):
        self.persen = persen

    def terapkan(self, harga):
        return harga * (1 - self.persen / 100)

class TransaksiDiskon:
    def __init__(self, transaksi, diskon):
        self.transaksi = transaksi
        self.diskon = diskon

    @property
    def omzet(self):
        return self.diskon.terapkan(self.transaksi.omzet)

td = TransaksiDiskon(t1, Diskon(10))
print(td.omzet)

Keluaran yang diharapkan:

```
229500.0
```

## 13. Penskala buatan sendiri

In [ ]:
class PenskalaSederhana:
    def fit(self, data):
        n = len(data)
        self.rerata_ = sum(data) / n
        ragam = sum((x - self.rerata_)**2 for x in data) / n
        self.simpangan_ = ragam ** 0.5
        return self

    def transform(self, data):
        if not hasattr(self, "rerata_"):
            raise RuntimeError("panggil fit() lebih dahulu")
        return [(x - self.rerata_) / self.simpangan_
                for x in data]

    def fit_transform(self, data):
        return self.fit(data).transform(data)

latih = [170, 180, 160, 175, 165]
uji = [172, 168]

s = PenskalaSederhana().fit(latih)
print(s.rerata_, round(s.simpangan_, 4))
print([round(v, 4) for v in s.transform(latih)])
print([round(v, 4) for v in s.transform(uji)])

Keluaran yang diharapkan:

```
170.0 7.0711
[0.0, 1.4142, -1.4142, 0.7071, -0.7071]
[0.2828, -0.2828]
```